# 🧊 SAR Sea Ice — Baseline Comparison (for Information Fusion)

Compares your model against published language-/reasoning-segmentation models,
**all evaluated on the same 90 SAR test images**, with identical metric code
(gIoU / cIoU / Dice). Mirrors Table 1 of the underwater reasoning-seg paper.

### ⚠️ How to use this notebook
These models need **incompatible `transformers` versions**, so they **cannot**
all run in one session. The workflow is:

1. **Run Section 0** (shared setup) — always, after every restart.
2. Run **one model section** at a time. Each saves its result to a JSON on Drive.
3. For the heavy models (LISA, GeoPixel, SEEM): **restart the runtime**
   (`Runtime → Restart session`), re-run Section 0, then run that model's section.
4. When all desired sections are done, run **Section 8 (Aggregate)** to build
   the comparison table + LaTeX from whatever JSONs exist on Drive.

Every number is from real inference on your data — nothing is copied from other
papers (those used different datasets, so copying would be invalid).

| Section | Model | Type | transformers | Session |
|---|---|---|---|---|
| 1 | **Your model** | SAR-trained, text-guided | new | shared |
| 2 | CLIPSeg | zero-shot text→mask | new | shared |
| 3 | lang-sam (GroundingDINO+SAM) | zero-shot | new | shared |
| 4 | DeepLabv3+ | trained, no text | new | shared |
| 5 | LISA-7B | zero-shot reasoning | **old (~4.31)** | **own** |
| 6 | GeoPixel | zero-shot RS-LMM | pinned | **own** |
| 7 | SEEM | zero-shot | pinned | **own** |
| 8 | **Aggregate → table + LaTeX** | — | any | shared |


## 0. 🔧 Shared Setup — RUN THIS FIRST (after every restart)

In [ ]:
# Mount Drive, clone repo, define the test-set lister + metric harness.
import os, subprocess, sys, json
import numpy as np
from pathlib import Path
from PIL import Image

# ── Drive ────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/sea_ice_seg'
except Exception as e:
    print('Drive mount skipped:', e)
    DRIVE_DIR = '/content/sea_ice_seg_drive'

COMPARE_DIR = f'{DRIVE_DIR}/outputs/comparison'
os.makedirs(COMPARE_DIR, exist_ok=True)
print('Comparison results dir:', COMPARE_DIR)

# ── Repo ─────────────────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/prakhar443/sea_ice_seg.git'
BRANCH   = 'claude/laughing-thompson-AhCX7'
REPO_DIR = '/content/sea_ice_seg'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# ── Install repo dependencies + fix Colab's torchao/peft conflict ─────────────
# Colab ships torchao 0.10.0, but peft demands torchao>=0.16.0 (which needs a
# newer torch than Colab has, so it won't upgrade). The pipeline never uses
# torchao — peft only probes for it. UNINSTALLING torchao makes peft skip the
# version check gracefully (is_torchao_available() returns False), avoiding the
# ImportError. This is more reliable than trying to upgrade torchao.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers>=4.40.0','peft>=0.10.0','accelerate>=0.27.0',
    'einops>=0.7.0','timm>=0.9.0','scikit-learn','pandas'], check=False)

# Belt-and-suspenders: if torchao is still importable in THIS kernel from a
# prior import, hide it so peft's find_spec check returns None.
import importlib.util
if importlib.util.find_spec('torchao') is not None:
    print('⚠️  torchao still present — if peft still errors, RESTART the runtime '
          'and run this cell again before Section 1.')

DATA_ROOT = f'{REPO_DIR}/dataset'

# ── Deterministic test-set lister (matches the training split exactly) ────────
def list_test_set(data_root=DATA_ROOT, seed=42, train_ratio=0.70, val_ratio=0.15):
    from config import ICE_CLASSES, ICE_CLASS_TO_IDX
    data_root = Path(data_root); samples = []
    for cls in ICE_CLASSES:
        imgs_dir = data_root / cls / 'images'
        mask_dir = data_root / cls / 'masks'
        if not imgs_dir.exists():
            continue
        files = sorted(imgs_dir.glob('*.jpg'))
        rng = np.random.default_rng(seed + ICE_CLASS_TO_IDX[cls])
        order = rng.permutation(len(files)).tolist()
        n = len(files); ntr = round(n*train_ratio); nv = round(n*val_ratio)
        for i in order[ntr+nv:]:
            # Mask naming convention: '1_1003_.jpg' -> '1_1003_scat.jpg'
            stem = files[i].stem.rstrip('_') + '_scat'
            mp = mask_dir / (stem + files[i].suffix)
            if not mp.exists():
                mp = mask_dir / files[i].name      # fallback
            samples.append((files[i], mp, cls))
    return samples

# ── Metric harness — identical for every model ────────────────────────────────
# predict_fn(pil_rgb) must return a HxW boolean/0-1 numpy mask (any resolution).
# Query for all text models: "sea ice".
def evaluate_predictor(predict_fn, eval_size=512, verbose=True):
    samples = list_test_set()
    inter_tot = union_tot = 0.0
    ious, dices = [], []
    for k, (img_path, mask_path, cls) in enumerate(samples):
        pil = Image.open(img_path).convert('RGB')
        pred = np.asarray(predict_fn(pil))
        if pred.dtype != bool:
            pred = pred > 0.5
        pred = np.array(Image.fromarray((pred.astype('uint8')*255))
                        .resize((eval_size, eval_size), Image.NEAREST)) > 127
        if mask_path.exists():
            from data.dataset import binarize_mask
            g  = np.array(Image.open(mask_path).convert('L'))
            gb = binarize_mask(g, mode='otsu')          # per-image Otsu {0,1}
            gt = np.array(Image.fromarray((gb*255).astype('uint8'))
                          .resize((eval_size, eval_size), Image.NEAREST)) > 127
        else:
            gt = np.zeros((eval_size, eval_size), bool)
        inter = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        iou  = inter/union if union > 0 else (1.0 if pred.sum()==0 else 0.0)
        dice = 2*inter/(pred.sum()+gt.sum()) if (pred.sum()+gt.sum())>0 else 1.0
        ious.append(iou); dices.append(dice)
        inter_tot += inter; union_tot += union
        if verbose and (k+1) % 15 == 0:
            print(f'  {k+1}/{len(samples)}  running gIoU={np.mean(ious):.4f}')
    return {'gIoU': float(np.mean(ious)),
            'cIoU': float(inter_tot/union_tot) if union_tot > 0 else 0.0,
            'Dice': float(np.mean(dices)),
            'n': len(samples)}

def save_result(name, res):
    p = f'{COMPARE_DIR}/{name}.json'
    json.dump({'model': name, **res}, open(p, 'w'), indent=2)
    print(f'\n✅ {name}: gIoU={res["gIoU"]:.4f}  cIoU={res["cIoU"]:.4f}  '
          f'Dice={res["Dice"]:.4f}  (n={res["n"]})  → saved {p}')

print('\n✅ Section 0 ready. Test images:', len(list_test_set()))


## 1. 🏆 Your Model (SAR-trained, text-guided)

Runs your trained baseline pipeline (`best_model.pth`) through the **same**
harness so its gIoU/cIoU/Dice are directly comparable to every baseline.


In [ ]:
import torch, numpy as np
from PIL import Image

# Locate the trained checkpoint
CKPT = None
for c in [f'{DRIVE_DIR}/outputs/best_model.pth', f'{REPO_DIR}/outputs/best_model.pth']:
    if os.path.exists(c): CKPT = c; break
assert CKPT, 'best_model.pth not found in Drive/outputs or repo/outputs'
print('Checkpoint:', CKPT)

from config import cfg
from models.pipeline import SeaIceSegmentationPipeline
from data.preprocessing import SARPreprocessor

device = torch.device('cuda')

# Load checkpoint first and auto-detect the U-Net base width it was trained with.
# config.py defaults to decoder_base_channels=32, but the A100 run used 48.
# Reading mask_decoder.enc1.0.weight's out-channels gives the exact base.
ck = torch.load(CKPT, map_location='cpu')
sd = ck.get('model_state_dict', ck)
if 'mask_decoder.enc1.0.weight' in sd:
    base = sd['mask_decoder.enc1.0.weight'].shape[0]
    cfg.model.decoder_base_channels = int(base)
    print(f'Detected U-Net base width from checkpoint: {base}')

model = SeaIceSegmentationPipeline(cfg.model).to(device).eval()
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f'Loaded checkpoint: {len(missing)} missing, {len(unexpected)} unexpected keys')

pre = SARPreprocessor(cfg.data)
GENERIC_TEXT = 'sea ice in synthetic aperture radar image'

@torch.no_grad()
def predict_ours(pil):
    # Match training EXACTLY: dataset feeds (gray/255.0) numpy to SARPreprocessor
    # (data/dataset.py line ~402), then forward(images, descriptions).
    arr = np.array(pil.convert('L')).astype('float32') / 255.0
    img = pre(arr).unsqueeze(0).to(device)
    out = model(img, [GENERIC_TEXT])
    m = out['masks'] if 'masks' in out else torch.sigmoid(out['mask_logits'])
    return m.squeeze().float().cpu().numpy() > 0.5

res = evaluate_predictor(predict_ours)
save_result('OursTextGuided', res)


## 2. CLIPSeg (zero-shot text→mask)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.40.0'])

import torch, numpy as np
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
device = torch.device('cuda')
_proc = CLIPSegProcessor.from_pretrained('CIDAS/clipseg-rd64-refined')
_cs   = CLIPSegForImageSegmentation.from_pretrained('CIDAS/clipseg-rd64-refined').to(device).eval()

@torch.no_grad()
def predict_clipseg(pil):
    inp = _proc(text=['sea ice'], images=[pil], return_tensors='pt').to(device)
    logits = _cs(**inp).logits           # (352,352)
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    return prob > 0.5

res = evaluate_predictor(predict_clipseg)
save_result('CLIPSeg', res)


## 3. lang-sam — GroundingDINO + SAM (zero-shot)

Represents the language-prompted Segment-Anything family (= Grounded-SAM).


In [ ]:
# GroundingDINO + SAM via transformers (robust; same method as lang-sam).
# The lang-sam PyPI package has a fragile dep tree on Colab, so we build the
# identical GroundingDINO->boxes->SAM->masks pipeline with pure transformers.
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.40.0'])

import torch, numpy as np
from transformers import (AutoProcessor, AutoModelForZeroShotObjectDetection,
                          SamModel, SamProcessor)
device = torch.device('cuda')

_gd_proc  = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-tiny')
_gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    'IDEA-Research/grounding-dino-tiny').to(device).eval()
_sam_proc  = SamProcessor.from_pretrained('facebook/sam-vit-base')
_sam_model = SamModel.from_pretrained('facebook/sam-vit-base').to(device).eval()

@torch.no_grad()
def predict_langsam(pil):
    # 1) GroundingDINO: detect "sea ice" boxes (text must be lowercase, end with '.')
    inp = _gd_proc(images=pil, text='sea ice.', return_tensors='pt').to(device)
    out = _gd_model(**inp)
    # transformers renamed box_threshold -> threshold across versions; try both.
    try:
        res = _gd_proc.post_process_grounded_object_detection(
            out, inp.input_ids, threshold=0.25, text_threshold=0.25,
            target_sizes=[pil.size[::-1]])
    except TypeError:
        res = _gd_proc.post_process_grounded_object_detection(
            out, inp.input_ids, box_threshold=0.25, text_threshold=0.25,
            target_sizes=[pil.size[::-1]])
    boxes = res[0]['boxes']
    if boxes.numel() == 0:
        return np.zeros((pil.height, pil.width), bool)   # no detection (honest zero-shot result)
    # 2) SAM: segment from the detected boxes
    sinp = _sam_proc(pil, input_boxes=[boxes.cpu().tolist()], return_tensors='pt').to(device)
    sout = _sam_model(**sinp)
    masks = _sam_proc.image_processor.post_process_masks(
        sout.pred_masks.cpu(), sinp['original_sizes'].cpu(),
        sinp['reshaped_input_sizes'].cpu())[0]
    masks = np.asarray(masks).astype(bool)               # (n_boxes, n_per_box, H, W)
    if masks.ndim == 4:
        masks = masks[:, 0]                              # first mask per box
    return np.any(masks, axis=0)                         # union of all instances

res = evaluate_predictor(predict_langsam)
save_result('LangSAM', res)


## 4. DeepLabv3+ (trained on SAR, no text)

Standard semantic-segmentation baseline with **no** language input. Trained
briefly on your SAR train split, evaluated with the same harness. Shows what a
strong vision-only model achieves without text guidance.


In [ ]:
import torch, torch.nn as nn, numpy as np
import torch.nn.functional as F
from torchvision.models.segmentation import deeplabv3_resnet50
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device('cuda')

class _SegDS(Dataset):
    def __init__(self, split):
        from config import ICE_CLASSES, ICE_CLASS_TO_IDX
        from pathlib import Path
        self.items = []
        dr = Path(DATA_ROOT)
        for cls in ICE_CLASSES:
            idir, mdir = dr/cls/'images', dr/cls/'masks'
            if not idir.exists(): continue
            files = sorted(idir.glob('*.jpg'))
            rng = np.random.default_rng(42 + ICE_CLASS_TO_IDX[cls])
            order = rng.permutation(len(files)).tolist()
            n=len(files); ntr=round(n*0.7); nv=round(n*0.15)
            chosen = order[:ntr] if split=='train' else order[ntr+nv:]
            for i in chosen: self.items.append((files[i], mdir/files[i].name))
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        ip, mp = self.items[k]
        a = np.asarray(Image.open(ip).convert('RGB').resize((512,512))).astype('float32')/255.0
        a = (a-a.min())/(a.max()-a.min()+1e-8)
        x = torch.from_numpy(a.transpose(2,0,1))
        m = (np.array(Image.open(mp).convert('L').resize((512,512), Image.NEAREST))>127).astype('float32')             if mp.exists() else np.zeros((512,512),'float32')
        return x, torch.from_numpy(m).unsqueeze(0)

dl = DataLoader(_SegDS('train'), batch_size=4, shuffle=True, num_workers=2)
net = deeplabv3_resnet50(num_classes=1).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)

print('Training DeepLabv3+ (15 epochs, no text)...')
net.train()
for ep in range(15):
    tot=0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        logit = net(x)['out']
        loss = F.binary_cross_entropy_with_logits(logit, y)              + (1 - (2*(torch.sigmoid(logit)*y).sum()+1)/((torch.sigmoid(logit)+y).sum()+1))
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    print(f'  epoch {ep+1}/15  loss={tot/len(dl):.4f}')

net.eval()
@torch.no_grad()
def predict_deeplab(pil):
    a = np.asarray(pil.resize((512,512))).astype('float32')/255.0
    a = (a-a.min())/(a.max()-a.min()+1e-8)
    x = torch.from_numpy(a.transpose(2,0,1)).unsqueeze(0).to(device)
    return torch.sigmoid(net(x)['out']).squeeze().cpu().numpy() > 0.5

res = evaluate_predictor(predict_deeplab)
save_result('DeepLabV3plus', res)


## 5a. 🐍 Python 3.10 Environment (for LISA / GeoPixel / SEEM)

⚠️ **Run in a FRESH session, after Section 0.** The heavy models pin 2023-era
deps that won't build on Colab's Python 3.12. This cell uses `uv` to create an
isolated **Python 3.10** environment; the models then run as a **subprocess** in
that env and write their result JSON to Drive, which Section 8 aggregates.

Run this **once per session** before Section 5/6/7. Takes ~3–5 min.


In [ ]:
# Build an isolated Python 3.10 env with uv (auto-downloads standalone 3.10).
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)

PY310_ENV = '/content/py310'
PYBIN     = f'{PY310_ENV}/bin/python'
if not os.path.exists(PYBIN):
    subprocess.run(['uv','venv','--python','3.10',PY310_ENV], check=True)

# torch cu118 (compatible with Colab's CUDA driver) + LISA-era deps.
# These wheels exist for cp310 (they do NOT for cp312 — that's why 3.12 failed).
subprocess.run(['uv','pip','install','--python',PYBIN,
    'torch==2.0.1','torchvision==0.15.2',
    '--index-url','https://download.pytorch.org/whl/cu118'], check=True)
# NOTE: bitsandbytes intentionally OMITTED. LISA runs in bf16 (no quantization),
# and bitsandbytes' prebuilt CUDA binary can't find libcusparse.so.11 in the
# cu118 torch wheel. accelerate guards its bnb import, so omitting it avoids the crash.
subprocess.run(['uv','pip','install','--python',PYBIN,
    'transformers==4.31.0','tokenizers==0.13.3','sentencepiece','peft==0.4.0',
    'einops','timm','opencv-python-headless',
    'accelerate==0.21.0','scipy','numpy<2','pillow'], check=True)

print('✅ Python 3.10 env ready:', PYBIN)
subprocess.run([PYBIN,'-c','import sys,torch;print(sys.version);print("torch",torch.__version__,"cuda",torch.cuda.is_available())'])


## 5. LISA-7B (zero-shot reasoning segmentation)

⚠️ **Run in a FRESH session** (`Runtime → Restart session`, then re-run Section 0).
LISA pins an **older `transformers`** that conflicts with CLIPSeg/your model.

Query: *"Where is the sea ice in this synthetic aperture radar image? Please
output the segmentation mask."*

> LISA's API can vary by commit. If a call signature differs, the error will be
> explicit — fix the one line and re-run. We never fabricate the mask.


In [ ]:
# Write a self-contained LISA eval script and run it in the Python 3.10 env.
# The script: clones LISA, loads xinlai/LISA-7B-v1, runs the same test split,
# computes gIoU/cIoU/Dice with identical formulas, writes the result JSON.
import os, subprocess, textwrap

LISA_SCRIPT = '/content/lisa_eval.py'
script = textwrap.dedent(r"""
    import os, sys, json, argparse
    import numpy as np
    from pathlib import Path
    from PIL import Image
    import cv2, torch

    ICE_CLASSES = ["Young Ice","First Year Ice","Floating Ice","Glaciers","Icebergs","Old Ice"]

    def otsu(gray):
        hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
        total = gray.size; sum_total = np.dot(np.arange(256), hist)
        sb=wb=mv=0.0; thr=127
        for t in range(256):
            wb += hist[t]
            if wb==0: continue
            wf = total-wb
            if wf==0: break
            sb += t*hist[t]; mb=sb/wb; mf=(sum_total-sb)/wf
            v = wb*wf*(mb-mf)**2
            if v>mv: mv=v; thr=t
        b=(gray>thr).astype(np.uint8); fg=b.mean()
        if fg<0.005 or fg>0.995: b=(gray>gray.mean()).astype(np.uint8)
        return b

    def list_test(data_root, seed=42, tr=0.70, vr=0.15):
        dr=Path(data_root); out=[]
        for ci,cls in enumerate(ICE_CLASSES):
            idir=dr/cls/'images'; mdir=dr/cls/'masks'
            if not idir.exists(): continue
            files=sorted(idir.glob('*.jpg'))
            rng=np.random.default_rng(seed+ci); order=rng.permutation(len(files)).tolist()
            n=len(files); ntr=round(n*tr); nv=round(n*vr)
            for i in order[ntr+nv:]:
                stem=files[i].stem.rstrip('_')+'_scat'
                mp=mdir/(stem+files[i].suffix)
                if not mp.exists(): mp=mdir/files[i].name
                out.append((files[i],mp,cls))
        return out

    def main():
        ap=argparse.ArgumentParser()
        ap.add_argument('--data_root',required=True); ap.add_argument('--out',required=True)
        ap.add_argument('--lisa_repo',default='/content/LISA')
        ap.add_argument('--version',default='xinlai/LISA-7B-v1'); ap.add_argument('--eval_size',type=int,default=512)
        a=ap.parse_args()
        sys.path.insert(0,a.lisa_repo)

        from transformers import AutoTokenizer, CLIPImageProcessor
        from model.LISA import LISAForCausalLM
        from model.llava import conversation as conversation_lib
        from model.llava.mm_utils import tokenizer_image_token
        from model.llava.constants import DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
        from model.segment_anything.utils.transforms import ResizeLongestSide

        tok=AutoTokenizer.from_pretrained(a.version,cache_dir=None,model_max_length=512,use_fast=False)
        tok.pad_token=tok.unk_token
        seg_idx=tok('[SEG]',add_special_tokens=False).input_ids[0]
        model=LISAForCausalLM.from_pretrained(a.version,torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,vision_tower='openai/clip-vit-large-patch14',
                seg_token_idx=seg_idx)
        model.get_model().initialize_vision_modules(model.get_model().config)
        vt=model.get_model().get_vision_tower(); vt.to(dtype=torch.bfloat16,device='cuda')
        model=model.bfloat16().cuda().eval()
        clip_proc=CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14')
        tf=ResizeLongestSide(1024)

        px_mean=torch.tensor([123.675,116.28,103.53]).view(-1,1,1).cuda()
        px_std =torch.tensor([58.395,57.12,57.375]).view(-1,1,1).cuda()
        def sam_pre(x):
            x=(x-px_mean)/px_std
            h,w=x.shape[-2:]
            x=torch.nn.functional.pad(x,(0,1024-w,0,1024-h))
            return x

        PROMPT=('Where is the sea ice in this synthetic aperture radar image? '
                'Please output segmentation mask.')
        samples=list_test(a.data_root)
        ious=[]; dices=[]; it=ut=0.0
        for k,(ip,mp,cls) in enumerate(samples):
            conv=conversation_lib.conv_templates['llava_v1'].copy()
            q=DEFAULT_IMAGE_TOKEN+'\n'+PROMPT
            conv.append_message(conv.roles[0],q); conv.append_message(conv.roles[1],None)
            prompt=conv.get_prompt()
            img_np=cv2.cvtColor(cv2.imread(str(ip)),cv2.COLOR_BGR2RGB)
            H,W=img_np.shape[:2]
            image_clip=clip_proc.preprocess(img_np,return_tensors='pt')['pixel_values'][0].unsqueeze(0).cuda().bfloat16()
            im=tf.apply_image(img_np); rl=[im.shape[:2]]
            im=sam_pre(torch.from_numpy(im).permute(2,0,1).contiguous().cuda().float()).unsqueeze(0).bfloat16()
            ids=tokenizer_image_token(prompt,tok,return_tensors='pt').unsqueeze(0).cuda()
            with torch.no_grad():
                out_ids,pred_masks=model.evaluate(image_clip,im,ids,rl,
                    original_size_list=[(H,W)],max_new_tokens=64,tokenizer=tok)
            pm=pred_masks[0].detach().cpu().numpy()
            pm=(pm[0] if pm.ndim==3 else pm)>0
            pred=np.array(Image.fromarray((pm.astype('uint8')*255)).resize((a.eval_size,a.eval_size),Image.NEAREST))>127
            if mp.exists():
                g=np.array(Image.open(mp).convert('L')); gb=otsu(g)
                gt=np.array(Image.fromarray((gb*255).astype('uint8')).resize((a.eval_size,a.eval_size),Image.NEAREST))>127
            else:
                gt=np.zeros((a.eval_size,a.eval_size),bool)
            inter=np.logical_and(pred,gt).sum(); union=np.logical_or(pred,gt).sum()
            iou=inter/union if union>0 else (1.0 if pred.sum()==0 else 0.0)
            dice=2*inter/(pred.sum()+gt.sum()) if (pred.sum()+gt.sum())>0 else 1.0
            ious.append(iou); dices.append(dice); it+=inter; ut+=union
            if (k+1)%15==0: print(f'  {k+1}/{len(samples)} gIoU={np.mean(ious):.4f}',flush=True)
        res={'model':'LISA7B','gIoU':float(np.mean(ious)),
             'cIoU':float(it/ut) if ut>0 else 0.0,'Dice':float(np.mean(dices)),'n':len(samples)}
        json.dump(res,open(a.out,'w'),indent=2)
        print('SAVED',a.out,res,flush=True)

    if __name__=='__main__': main()
""")
open(LISA_SCRIPT,'w').write(script)

# Ensure bitsandbytes is absent (its broken CUDA import crashes the chain).
subprocess.run(['uv','pip','uninstall','--python',PYBIN,'bitsandbytes'], check=False)

if not os.path.exists('/content/LISA'):
    subprocess.run(['git','clone','https://github.com/dvlab-research/LISA','/content/LISA'], check=True)

# bitsandbytes (imported via transformers->accelerate) needs the CUDA libs that
# the torch cu118 wheel ships inside the env's nvidia/* and torch/lib folders.
# bitsandbytes only searches LD_LIBRARY_PATH, so add those dirs for the subprocess.
import glob
SP = f'{PY310_ENV}/lib/python3.10/site-packages'
lib_dirs = glob.glob(f'{SP}/nvidia/*/lib') + [f'{SP}/torch/lib', '/usr/local/cuda/lib64']
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [env.get('LD_LIBRARY_PATH', '')])

print('Running LISA in the Python 3.10 env (this loads a 7B model; be patient)...')
r = subprocess.run([PYBIN, LISA_SCRIPT,
    '--data_root', DATA_ROOT, '--out', f'{COMPARE_DIR}/LISA7B.json',
    '--lisa_repo', '/content/LISA'], capture_output=True, text=True, env=env)

print('================= STDOUT (tail) =================')
print(r.stdout[-4000:])
print('================= STDERR (tail) =================')
print(r.stderr[-6000:])
print('================================================')
if r.returncode == 0:
    import json
    print('\n✅', json.load(open(f'{COMPARE_DIR}/LISA7B.json')))
else:
    print(f'\n⚠️  LISA exited with code {r.returncode}. Copy the STDERR block above to me.')


## 6. GeoPixel (zero-shot remote-sensing LMM)

⚠️ **Run in a FRESH session, after Section 0.** GeoPixel needs **transformers
4.33.2** (newer than LISA's 4.31) and vendored SAM2, so it gets its **own**
Python 3.10 env. The model runs as a subprocess and writes `GeoPixel.json`.

Closest-domain baseline (remote sensing). Uses the GeoPixel-7B-RES referring
variant with the query "Please segment the sea ice in this image."


In [ ]:
# GeoPixel in its own Python 3.10 env (transformers 4.33.2 + vendored SAM2).
import os, sys, subprocess, textwrap, glob

# Clone the repo (provides model/geopixel.py + vendored sam2 code)
if not os.path.exists('/content/GeoPixel'):
    subprocess.run(['git','clone','https://github.com/mbzuai-oryx/GeoPixel','/content/GeoPixel'], check=True)

# ── Build dedicated env ───────────────────────────────────────────────────────
subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=False)
GP_ENV  = '/content/py310_gp'
GP_PY   = f'{GP_ENV}/bin/python'
if not os.path.exists(GP_PY):
    subprocess.run(['uv','venv','--python','3.10',GP_ENV], check=True)

# torch 2.1.2 cu118 (compatible with Colab driver + SAM2/InternLM), then GeoPixel deps.
subprocess.run(['uv','pip','install','--python',GP_PY,
    'torch==2.1.2','torchvision==0.16.2',
    '--index-url','https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run(['uv','pip','install','--python',GP_PY,
    'transformers==4.33.2','accelerate==0.34.2','peft==0.8.2','einops==0.8.0',
    'opencv-python-headless==4.10.0.84','sentencepiece==0.1.99','numpy==1.26.4',
    'pycocotools==2.0.8','hydra-core==1.3.2','timm','scipy','pillow',
    'matplotlib==3.9.2','decord==0.6.0','tensorboard==2.18.0'], check=True)
# deepspeed is imported by some InternLM code paths; install (has wheels). Non-fatal.
subprocess.run(['uv','pip','install','--python',GP_PY,'deepspeed==0.12.3'], check=False)

# GeoPixel's InternLM-XComposer LLM requires flash-attn for its attention layers.
# Install a PREBUILT wheel matching this env (torch2.1 + cu118 + cp310, abiFALSE
# to match pytorch.org wheels) so we avoid a 20+ min source build.
FA_WHL = ('https://github.com/Dao-AILab/flash-attention/releases/download/'
          'v2.5.8/flash_attn-2.5.8+cu118torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl')
subprocess.run(['uv','pip','install','--python',GP_PY,FA_WHL], check=False)
print('✅ GeoPixel env ready:', GP_PY)

# ── Self-contained eval script ────────────────────────────────────────────────
GP_SCRIPT = '/content/geopixel_eval.py'
script = textwrap.dedent(r"""
    import os, sys, json, argparse
    import numpy as np
    from pathlib import Path
    from PIL import Image
    import torch, transformers

    ICE_CLASSES = ["Young Ice","First Year Ice","Floating Ice","Glaciers","Icebergs","Old Ice"]

    def otsu(gray):
        hist=np.bincount(gray.ravel(),minlength=256).astype(np.float64)
        total=gray.size; st=np.dot(np.arange(256),hist); sb=wb=mv=0.0; thr=127
        for t in range(256):
            wb+=hist[t]
            if wb==0: continue
            wf=total-wb
            if wf==0: break
            sb+=t*hist[t]; mb=sb/wb; mf=(st-sb)/wf; v=wb*wf*(mb-mf)**2
            if v>mv: mv=v; thr=t
        b=(gray>thr).astype(np.uint8); fg=b.mean()
        if fg<0.005 or fg>0.995: b=(gray>gray.mean()).astype(np.uint8)
        return b

    def list_test(dr,seed=42,tr=0.70,vr=0.15):
        dr=Path(dr); out=[]
        for ci,cls in enumerate(ICE_CLASSES):
            idir=dr/cls/'images'; mdir=dr/cls/'masks'
            if not idir.exists(): continue
            files=sorted(idir.glob('*.jpg'))
            rng=np.random.default_rng(seed+ci); order=rng.permutation(len(files)).tolist()
            n=len(files); ntr=round(n*tr); nv=round(n*vr)
            for i in order[ntr+nv:]:
                stem=files[i].stem.rstrip('_')+'_scat'; mp=mdir/(stem+files[i].suffix)
                if not mp.exists(): mp=mdir/files[i].name
                out.append((files[i],mp,cls))
        return out

    def main():
        ap=argparse.ArgumentParser()
        ap.add_argument('--data_root',required=True); ap.add_argument('--out',required=True)
        ap.add_argument('--repo',default='/content/GeoPixel')
        ap.add_argument('--version',default='MBZUAI/GeoPixel-7B-RES')
        ap.add_argument('--eval_size',type=int,default=512)
        a=ap.parse_args(); sys.path.insert(0,a.repo)
        from model.geopixel import GeoPixelForCausalLM

        tok=transformers.AutoTokenizer.from_pretrained(a.version,cache_dir=None,
            padding_side='right',use_fast=False,trust_remote_code=True)
        tok.pad_token=tok.unk_token
        seg_i,bop_i,eop_i=[tok(t,add_special_tokens=False).input_ids[0] for t in ['[SEG]','<p>','</p>']]
        gargs={'vision_pretrained':'facebook/sam2-hiera-large','seg_token_idx':seg_i,
               'bop_token_idx':bop_i,'eop_token_idx':eop_i}
        model=GeoPixelForCausalLM.from_pretrained(a.version,low_cpu_mem_usage=True,
               torch_dtype=torch.bfloat16,**gargs)
        # Required setup from chat.py: evaluate() uses self.tokenizer internally.
        model.config.eos_token_id=tok.eos_token_id
        model.config.bos_token_id=tok.bos_token_id
        model.config.pad_token_id=tok.pad_token_id
        model.tokenizer=tok
        model=model.bfloat16().cuda().eval()

        QUERY='Please segment the sea ice in this image.'
        samples=list_test(a.data_root); ious=[]; dices=[]; it=ut=0.0
        for k,(ip,mp,cls) in enumerate(samples):
            try:
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    resp,pred_masks=model.evaluate(tok,QUERY,images=[str(ip)],max_new_tokens=300)
            except Exception as e:
                print('  eval err',ip.name,repr(e)[:120],flush=True); pred_masks=None
            H=W=a.eval_size
            if pred_masks is not None and len(pred_masks)>0:
                pm=pred_masks[0].detach().cpu().numpy()
                pm=(pm[0] if pm.ndim==3 else pm)>0
                pred=np.array(Image.fromarray((pm.astype('uint8')*255)).resize((W,H),Image.NEAREST))>127
            else:
                pred=np.zeros((H,W),bool)
            if mp.exists():
                g=np.array(Image.open(mp).convert('L')); gb=otsu(g)
                gt=np.array(Image.fromarray((gb*255).astype('uint8')).resize((W,H),Image.NEAREST))>127
            else:
                gt=np.zeros((H,W),bool)
            inter=np.logical_and(pred,gt).sum(); union=np.logical_or(pred,gt).sum()
            iou=inter/union if union>0 else (1.0 if pred.sum()==0 else 0.0)
            dice=2*inter/(pred.sum()+gt.sum()) if (pred.sum()+gt.sum())>0 else 1.0
            ious.append(iou); dices.append(dice); it+=inter; ut+=union
            if (k+1)%15==0: print(f'  {k+1}/{len(samples)} gIoU={np.mean(ious):.4f}',flush=True)
        res={'model':'GeoPixel','gIoU':float(np.mean(ious)),
             'cIoU':float(it/ut) if ut>0 else 0.0,'Dice':float(np.mean(dices)),'n':len(samples)}
        json.dump(res,open(a.out,'w'),indent=2); print('SAVED',a.out,res,flush=True)

    if __name__=='__main__': main()
""")
open(GP_SCRIPT,'w').write(script)

# CUDA libs for the subprocess
SP=f'{GP_ENV}/lib/python3.10/site-packages'
libs=glob.glob(f'{SP}/nvidia/*/lib')+[f'{SP}/torch/lib','/usr/local/cuda/lib64']
env=os.environ.copy(); env['LD_LIBRARY_PATH']=':'.join(libs+[env.get('LD_LIBRARY_PATH','')])
env['MPLBACKEND']='Agg'   # Colab's inline backend isn't valid in the py3.10 env

print('Running GeoPixel (downloads ~7B model + SAM2; be patient)...')
r=subprocess.run([GP_PY,GP_SCRIPT,'--data_root',DATA_ROOT,
    '--out',f'{COMPARE_DIR}/GeoPixel.json','--repo','/content/GeoPixel'],
    capture_output=True,text=True,env=env)
print('================= STDOUT (tail) ================='); print(r.stdout[-4000:])
print('================= STDERR (tail) ================='); print(r.stderr[-6000:])
print('================================================')
if r.returncode==0:
    import json; print('\n✅',json.load(open(f'{COMPARE_DIR}/GeoPixel.json')))
else:
    print(f'\n⚠️  GeoPixel exited with code {r.returncode}. Copy the STDERR block to me.')


## 7. SEEM (zero-shot, text-promptable)

⚠️ **Run in a FRESH session.** Heavy install (detectron2-style deps).
Segment-Everything-Everywhere with the text prompt "sea ice".


In [ ]:
# SEEM — zero-shot text-promptable segmentation in an isolated py3.10 env.
import os, sys, subprocess, textwrap, glob

# ── Clone SEEM v1.0 ───────────────────────────────────────────────────────────
if not os.path.exists('/content/SEEM'):
    subprocess.run(['git','clone','--depth','1','--branch','v1.0',
        'https://github.com/UX-Decoder/Segment-Everything-Everywhere-All-At-Once',
        '/content/SEEM'], check=True)

# ── Build dedicated py3.10 env ────────────────────────────────────────────────
subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=False)
SE_ENV = '/content/py310_seem'
SE_PY  = f'{SE_ENV}/bin/python'
if not os.path.exists(SE_PY):
    subprocess.run(['uv','venv','--python','3.10',SE_ENV], check=True)

# torch 2.0.1 cu118 (matches the detectron2 prebuilt wheel below)
subprocess.run(['uv','pip','install','--python',SE_PY,
    'torch==2.0.1','torchvision==0.15.2',
    '--index-url','https://download.pytorch.org/whl/cu118'], check=True)

# ── detectron2: try prebuilt wheel, fall back to minimal stubs ────────────────
D2_INDEX = 'https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/'
rd2 = subprocess.run(
    ['uv','pip','install','--python',SE_PY,'detectron2','--find-links',D2_INDEX],
    capture_output=True)
if rd2.returncode == 0:
    print('✅ detectron2 prebuilt wheel installed')
else:
    print('Prebuilt wheel unavailable; creating minimal detectron2 stubs for SEEM inference...')
    SP = f'{SE_ENV}/lib/python3.10/site-packages'
    stubs = {
        'detectron2/__init__.py': '',
        'detectron2/utils/__init__.py': '',
        'detectron2/utils/colormap.py': (
            'import numpy as np\n'
            '_C=[(220,20,60),(119,11,32),(0,0,142),(0,0,230),(106,0,228),(0,60,100)]*50\n'
            'def random_color(rgb=True,maximum=255):\n'
            '    c=np.random.randint(0,256,3); return tuple(int(x) for x in c)\n'
            'def colormap(rgb=True,maximum=255): return _C\n'
        ),
        'detectron2/utils/memory.py': (
            'def retry_if_cuda_oom(fn): return fn\n'
        ),
        'detectron2/data/__init__.py': (
            'class _M:\n'
            '    thing_classes=[]; thing_colors=[(220,20,60)]*200\n'
            '    stuff_classes=[]; stuff_colors=[(0,120,200)]*200\n'
            '    def set(self,**kw):\n'
            '        for k,v in kw.items(): setattr(self,k,v); return self\n'
            '    def get(self,k,d=None): return getattr(self,k,d)\n'
            'class MetadataCatalog:\n'
            '    _d={}\n'
            '    @classmethod\n'
            '    def get(cls,n):\n'
            '        if n not in cls._d: cls._d[n]=_M()\n'
            '        return cls._d[n]\n'
            '    @classmethod\n'
            '    def list(cls): return list(cls._d)\n'
        ),
        'detectron2/data/datasets/__init__.py': '',
        'detectron2/data/datasets/builtin_meta.py': 'COCO_CATEGORIES=[]\n',
        'detectron2/structures/__init__.py': (
            'import torch\n'
            'class BitMasks:\n'
            '    def __init__(self,t): self.tensor=torch.as_tensor(t) if not isinstance(t,torch.Tensor) else t\n'
            '    def __len__(self): return len(self.tensor)\n'
            '    def __getitem__(self,i): return BitMasks(self.tensor[i])\n'
            '    def to(self,dev): return BitMasks(self.tensor.to(dev))\n'
            '    def get_bounding_boxes(self):\n'
            '        n=len(self.tensor); b=torch.zeros(n,4)\n'
            '        for i,m in enumerate(self.tensor):\n'
            '            if m.any():\n'
            '                r=m.any(1).nonzero(); c=m.any(0).nonzero()\n'
            '                if len(r) and len(c): b[i]=torch.tensor([c[0,0],r[0,0],c[-1,0],r[-1,0]],dtype=torch.float32)\n'
            '        return b\n'
            'class Boxes:\n'
            '    def __init__(self,t): self.tensor=torch.as_tensor(t)\n'
            '    def __len__(self): return len(self.tensor)\n'
            '    def to(self,d): return Boxes(self.tensor.to(d))\n'
            'class Instances:\n'
            '    def __init__(self,sz,**kw): self.image_size=sz; self._f=dict(**kw)\n'
            '    def __setattr__(self,k,v):\n'
            '        if k in("image_size","_f"): super().__setattr__(k,v)\n'
            '        else: self._f[k]=v\n'
            '    def __getattr__(self,k):\n'
            '        if k in self._f: return self._f[k]\n'
            '        raise AttributeError(k)\n'
            '    def __len__(self): return len(next(iter(self._f.values()))) if self._f else 0\n'
            '    def has(self,k): return k in self._f\n'
            '    def to(self,d):\n'
            '        r=Instances(self.image_size)\n'
            '        for k,v in self._f.items(): r._f[k]=v.to(d) if hasattr(v,"to") else v\n'
            '        return r\n'
            'class ImageList:\n'
            '    def __init__(self,tensor,image_sizes): self.tensor=tensor; self.image_sizes=image_sizes\n'
            '    def __len__(self): return len(self.image_sizes)\n'
            '    def __getitem__(self,i):\n'
            '        h,w=self.image_sizes[i]; return self.tensor[i,...,:h,:w]\n'
            '    @property\n'
            '    def device(self): return self.tensor.device\n'
            '    def to(self,*a,**k): return ImageList(self.tensor.to(*a,**k),self.image_sizes)\n'
            '    @staticmethod\n'
            '    def from_tensors(tensors,size_divisibility=0,pad_value=0.0):\n'
            '        import math\n'
            '        image_sizes=[(int(t.shape[-2]),int(t.shape[-1])) for t in tensors]\n'
            '        mh=max(s[0] for s in image_sizes); mw=max(s[1] for s in image_sizes)\n'
            '        if size_divisibility>1:\n'
            '            mh=int(math.ceil(mh/size_divisibility)*size_divisibility)\n'
            '            mw=int(math.ceil(mw/size_divisibility)*size_divisibility)\n'
            '        b=tensors[0].new_full((len(tensors),tensors[0].shape[0],mh,mw),pad_value)\n'
            '        for i,t in enumerate(tensors): b[i,:,:t.shape[-2],:t.shape[-1]].copy_(t)\n'
            '        return ImageList(b,image_sizes)\n'
            'class BoxMode:\n'
            '    XYXY_ABS=0; XYWH_ABS=1; XYXY_REL=2; XYWH_REL=3; XYWHA_ABS=4\n'
        ),
        'detectron2/config/__init__.py': (
            'def configurable(f=None,*,from_config=None):\n'
            '    if f is not None: return f\n'
            '    def d(fn): return fn\n'
            '    return d\n'
            'class CfgNode(dict):\n'
            '    def __getattr__(self,k):\n'
            '        try: return self[k]\n'
            '        except KeyError: raise AttributeError(k)\n'
            '    def __setattr__(self,k,v): self[k]=v\n'
        ),
        'detectron2/layers/__init__.py': (
            'import torch.nn as nn\n'
            'import torch\n'
            'def cat(tensors,dim=0):\n'
            '    if len(tensors)==1: return tensors[0]\n'
            '    return torch.cat(tensors,dim)\n'
            'def shapes_to_tensor(x,device=None):\n'
            '    return torch.as_tensor(x,dtype=torch.long,device=device)\n'
            'Conv2d=nn.Conv2d\n'
            'Linear=nn.Linear\n'
            'def get_norm(norm,out_channels):\n'
            '    if norm=="GN": return nn.GroupNorm(32,out_channels)\n'
            '    if norm=="BN": return nn.BatchNorm2d(out_channels)\n'
            '    return nn.Identity()\n'
            'class ShapeSpec:\n'
            '    def __init__(self,*,channels=None,height=None,width=None,stride=None):\n'
            '        self.channels=channels; self.height=height; self.width=width; self.stride=stride\n'
        ),
        'detectron2/modeling/__init__.py': (
            'class Backbone: pass\n'
        ),
    }
    import os
    for rel_path, content in stubs.items():
        full = os.path.join(SP, rel_path)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        open(full, 'w').write(content)
    print('✅ detectron2 stubs written to', SP)

# ── SEEM inference deps ──────────────────────────────────────────────────────
# open_clip_torch and fvcore/iopath removed: conflict with timm pin.
# SEEM bundles UniCL text encoder; fvcore/iopath are detectron2 utils we stubbed.
# timm relaxed to >=0.6,<1: keeps the models.layers backward-compat API that
# FocalNet uses (from timm.models.layers import DropPath, trunc_normal_, etc.).
subprocess.run(['uv','pip','install','--python',SE_PY,
    'transformers==4.34.0','sentencepiece',
    'timm>=0.6.0,<1.0',
    'einops','omegaconf>=2.3.0','yacs',
    'ftfy','regex',
    'scipy','numpy>=1.23,<2','pillow','opencv-python-headless',
    'huggingface-hub','nltk'], check=True)
print('✅ SEEM env ready:', SE_PY)

# ── Self-contained eval script ────────────────────────────────────────────────
SE_SCRIPT = '/content/seem_eval.py'
script = textwrap.dedent(r"""
    import os, sys, json, argparse, warnings
    warnings.filterwarnings('ignore')
    import numpy as np
    from pathlib import Path
    from PIL import Image
    from scipy import ndimage
    import torch

    ICE_CLASSES = ["Young Ice","First Year Ice","Floating Ice","Glaciers","Icebergs","Old Ice"]

    def otsu(gray):
        hist=np.bincount(gray.ravel(),minlength=256).astype(np.float64)
        total=gray.size; st=np.dot(np.arange(256),hist); sb=wb=mv=0.0; thr=127
        for t in range(256):
            wb+=hist[t]
            if wb==0: continue
            wf=total-wb
            if wf==0: break
            sb+=t*hist[t]; mb=sb/wb; mf=(st-sb)/wf; v=wb*wf*(mb-mf)**2
            if v>mv: mv=v; thr=t
        b=(gray>thr).astype(np.uint8); fg=b.mean()
        if fg<0.005 or fg>0.995: b=(gray>gray.mean()).astype(np.uint8)
        return b

    def list_test(dr,seed=42,tr=0.70,vr=0.15):
        dr=Path(dr); out=[]
        for ci,cls in enumerate(ICE_CLASSES):
            idir=dr/cls/'images'; mdir=dr/cls/'masks'
            if not idir.exists(): continue
            files=sorted(idir.glob('*.jpg'))
            rng=np.random.default_rng(seed+ci); order=rng.permutation(len(files)).tolist()
            n=len(files); ntr=round(n*tr); nv=round(n*vr)
            for i in order[ntr+nv:]:
                stem=files[i].stem.rstrip('_')+'_scat'; mp=mdir/(stem+files[i].suffix)
                if not mp.exists(): mp=mdir/files[i].name
                out.append((files[i],mp,cls))
        return out

    def load_model(repo, ckpt_path):
        sys.path.insert(0, repo)
        sys.path.insert(0, os.path.join(repo,'demo'))
        sys.path.insert(0, os.path.join(repo,'demo','seem'))
        from modeling.BaseModel import BaseModel
        from modeling import build_model
        from utils.arguments import load_opt_from_config_files
        from utils.distributed import init_distributed
        cfg = os.path.join(repo,'configs','seem','focall_unicl_lang_v1.yaml')
        opt = load_opt_from_config_files([cfg])
        opt = init_distributed(opt)
        model = BaseModel(opt, build_model(opt)).from_pretrained(ckpt_path).cuda().eval()
        return model

    def get_mask(model, pil_img, repo, reftxt='sea ice'):
        # Run SEEM text-grounded segmentation; return binary mask via image diffing.
        sys.path.insert(0, os.path.join(repo,'demo','seem'))
        from tasks.interactive import interactive_infer_image

        img512 = pil_img.convert('RGB').resize((512,512), Image.BICUBIC)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                result = interactive_infer_image(
                    model, None, img512, ['Text'], reftxt=reftxt)

        # result = (PIL_visualisation, None)
        # SEEM overlays ~50% opacity colour onto masked pixels; diff reveals the mask.
        if result is None:
            return np.zeros((512,512), bool)
        viz = result[0] if isinstance(result,(list,tuple)) else result
        if viz is None:
            return np.zeros((512,512), bool)

        orig = np.array(img512).astype(np.float32)
        out_arr = np.array(viz.convert('RGB').resize((512,512))).astype(np.float32)
        diff = np.abs(orig - out_arr).max(axis=2)
        mask = diff > 15

        lbl, n = ndimage.label(mask)
        if n > 0:
            sizes = np.bincount(lbl.ravel()); sizes[0] = 0
            mask = lbl == sizes.argmax()
        return mask.astype(bool)

    def main():
        ap = argparse.ArgumentParser()
        ap.add_argument('--data_root', required=True)
        ap.add_argument('--out', required=True)
        ap.add_argument('--repo', default='/content/SEEM')
        ap.add_argument('--ckpt', default='/content/seem_focall_v1.pt')
        ap.add_argument('--eval_size', type=int, default=512)
        a = ap.parse_args()

        if not os.path.exists(a.ckpt):
            import urllib.request
            print('Downloading SEEM checkpoint (~700 MB)...', flush=True)
            urllib.request.urlretrieve(
                'https://huggingface.co/xdecoder/SEEM/resolve/main/seem_focall_v1.pt',
                a.ckpt)
            print('Downloaded.', flush=True)

        print('Loading SEEM...', flush=True)
        model = load_model(a.repo, a.ckpt)
        print('Model loaded.', flush=True)

        samples = list_test(a.data_root)
        ious=[]; dices=[]; it=ut=0.0

        for k,(ip,mp,cls) in enumerate(samples):
            try:
                pil = Image.open(ip).convert('RGB')
                pred = get_mask(model, pil, a.repo)
                pred = np.array(
                    Image.fromarray((pred*255).astype('uint8'))
                    .resize((a.eval_size, a.eval_size), Image.NEAREST)) > 127
            except Exception as e:
                print(f'  err [{k}] {ip.name}: {repr(e)[:100]}', flush=True)
                pred = np.zeros((a.eval_size, a.eval_size), bool)

            if mp.exists():
                g = np.array(Image.open(mp).convert('L'))
                gb = otsu(g)
                gt = np.array(Image.fromarray((gb*255).astype('uint8'))
                              .resize((a.eval_size, a.eval_size), Image.NEAREST)) > 127
            else:
                gt = np.zeros((a.eval_size, a.eval_size), bool)

            inter = np.logical_and(pred,gt).sum()
            union = np.logical_or(pred,gt).sum()
            iou  = inter/union if union>0 else (1.0 if pred.sum()==0 else 0.0)
            dice = 2*inter/(pred.sum()+gt.sum()) if (pred.sum()+gt.sum())>0 else 1.0
            ious.append(iou); dices.append(dice); it+=inter; ut+=union

            if (k+1)%15==0:
                print(f'  {k+1}/{len(samples)} gIoU={np.mean(ious):.4f}', flush=True)

        res = {'model':'SEEM', 'gIoU':float(np.mean(ious)),
               'cIoU':float(it/ut) if ut>0 else 0.0,
               'Dice':float(np.mean(dices)), 'n':len(samples)}
        json.dump(res, open(a.out,'w'), indent=2)
        print('SAVED', a.out, res, flush=True)

    if __name__=='__main__': main()
""")
open(SE_SCRIPT,'w').write(script)

# CUDA libs for the subprocess
SP2  = f'{SE_ENV}/lib/python3.10/site-packages'
libs = glob.glob(f'{SP2}/nvidia/*/lib')+[f'{SP2}/torch/lib','/usr/local/cuda/lib64']
env  = os.environ.copy()
env['LD_LIBRARY_PATH'] = ':'.join(libs+[env.get('LD_LIBRARY_PATH','')])
env['MPLBACKEND'] = 'Agg'

print('Running SEEM (downloads ~700 MB checkpoint on first run)...')
r = subprocess.run([SE_PY, SE_SCRIPT,
    '--data_root', DATA_ROOT,
    '--out', f'{COMPARE_DIR}/SEEM.json',
    '--repo', '/content/SEEM'],
    capture_output=True, text=True, env=env)
print('=============== STDOUT (tail) ==============='); print(r.stdout[-4000:])
print('=============== STDERR (tail) ==============='); print(r.stderr[-6000:])
print('=============================================')
if r.returncode==0:
    import json as _json; print('\n✅', _json.load(open(f'{COMPARE_DIR}/SEEM.json')))
else:
    print(f'\n⚠️  SEEM exited with code {r.returncode}. Copy the STDERR block to me.')


## 8. 📊 Aggregate → Comparison Table + LaTeX

Reads every `*.json` saved in the Drive comparison folder and builds the table.
Run this in any session after you've collected the model results you want.


In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob(f'{COMPARE_DIR}/*.json')):
    d = json.load(open(f))
    rows.append(d)

if not rows:
    print('No results yet. Run at least one model section.')
else:
    PRETTY = {'OursTextGuided':'Ours (text-guided)', 'CLIPSeg':'CLIPSeg',
              'LangSAM':'lang-sam (GD+SAM)', 'DeepLabV3plus':'DeepLabv3+ (no text)',
              'LISA7B':'LISA-7B', 'GeoPixel':'GeoPixel', 'SEEM':'SEEM'}
    df = pd.DataFrame(rows)
    df['Model'] = df['model'].map(lambda m: PRETTY.get(m, m))
    df = df[['Model','gIoU','cIoU','Dice','n']].copy()
    for c in ['gIoU','cIoU','Dice']:
        df[c] = (df[c]*100).round(2)
    # Order: baselines first, ours last (highlighted)
    df['_ours'] = df['Model'].str.startswith('Ours')
    df = df.sort_values(['_ours','gIoU']).drop(columns='_ours').reset_index(drop=True)
    print(df.to_string(index=False))

    # ── LaTeX (IEEE-style, caption above) ─────────────────────────────────────
    print('\n' + '='*70 + '\nLaTeX:\n' + '='*70)
    lines = [r'\begin{table}[t]', r'\centering',
             r'\caption{Comparison on the SAR sea ice test set ('
             + f'{int(df["n"].iloc[0])}' + r' images). All models evaluated with '
             r'identical metric code. gIoU/cIoU/Dice in \%.}',
             r'\label{tab:comparison}',
             r'\begin{tabular}{lccc}', r'\hline',
             r'Method & gIoU & cIoU & Dice \\', r'\hline']
    for _, r in df.iterrows():
        name = r['Model']
        if name.startswith('Ours'):
            lines.append(rf"\textbf{{{name}}} & \textbf{{{r.gIoU}}} & "
                         rf"\textbf{{{r.cIoU}}} & \textbf{{{r.Dice}}} \\")
        else:
            lines.append(rf"{name} & {r.gIoU} & {r.cIoU} & {r.Dice} \\")
    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    print('\n'.join(lines))

    df.to_csv(f'{COMPARE_DIR}/comparison_table.csv', index=False)
    print('\nSaved CSV →', f'{COMPARE_DIR}/comparison_table.csv')
